# WAXAL ASR Baseline - Whisper Small
**Lingala, Shona, Luganda** | Metric: 0.5xWER + 0.5xCER

In [1]:
# Redirect HF cache to D: drive (C: only has 0.3GB free)
import os
if False:
    os.environ['HF_HOME'] = 'D:\\hf_cache'
    os.environ['TRANSFORMERS_CACHE'] = 'D:\\hf_cache\\transformers'
    os.environ['HUGGINGFACE_HUB_CACHE'] = 'D:\\hf_cache\\hub'
    print('Cache set to D:\\hf_cache (102GB free)')
else:
    print('D:\\hf_cache not found. Create it if C: is full.')


Cache set to D:\hf_cache (102GB free)


In [2]:
!pip install -q torch torchaudio datasets transformers accelerate evaluate jiwer soundfile librosa pandas numpy tqdm torchcodec
import sys
if 'google.colab' in sys.modules:
    !apt-get -qq install ffmpeg
print('OK')


OK


In [3]:
import os, random, gc
import numpy as np
import torch
import pandas as pd
import librosa
from datasets import load_dataset, Dataset, interleave_datasets
from transformers import (
    WhisperFeatureExtractor, WhisperTokenizer, WhisperProcessor,
    WhisperForConditionalGeneration, Seq2SeqTrainingArguments, Seq2SeqTrainer
)
from evaluate import load as load_metric
import warnings
warnings.filterwarnings('ignore')

def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
set_seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

C:\Users\aby\AppData\Roaming\Python\Python313\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


Device: cpu


## 1. Load WAXAL (streaming = no disk/RAM issues)

In [4]:
target_langs = ['lin', 'sna', 'lug']
configs = [f'{lang}_asr' for lang in target_langs]
print(f'Loading: {configs}')

all_train, all_val, all_test = [], [], []
for config in configs:
    print(f'  {config}...')
    ds = load_dataset('google/WaxalNLP', config, streaming=True)
    all_train.append(ds['train'])
    all_val.append(ds['validation'])
    all_test.append(ds['test'])

train_ds = interleave_datasets(all_train)
val_ds = interleave_datasets(all_val)
test_ds = interleave_datasets(all_test)

NUM_TRAIN = 80
NUM_VAL = 10
train_iter = train_ds.shuffle(seed=42).take(NUM_TRAIN)
val_iter = val_ds.shuffle(seed=42).take(NUM_VAL)
print(f'Train: {NUM_TRAIN}, Val: {NUM_VAL}')

Loading: ['lin_asr', 'sna_asr', 'lug_asr']
  lin_asr...


Resolving data files:   0%|          | 0/72 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/40 [00:00<?, ?it/s]

  sna_asr...


Resolving data files:   0%|          | 0/72 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/52 [00:00<?, ?it/s]

  lug_asr...


Resolving data files:   0%|          | 0/72 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/67 [00:00<?, ?it/s]

Train: 80, Val: 20


## 2. Setup Whisper

In [5]:
model_name = 'openai/whisper-base'
cache_dir = 'D:/hf_cache'
processor = WhisperProcessor.from_pretrained(model_name, language='en', task='transcribe', cache_dir=cache_dir)
feature_extractor = WhisperFeatureExtractor.from_pretrained(model_name, cache_dir=cache_dir)
tokenizer = WhisperTokenizer.from_pretrained(model_name, language='en', task='transcribe', cache_dir=cache_dir)
model = WhisperForConditionalGeneration.from_pretrained(model_name, cache_dir=cache_dir, low_cpu_mem_usage=True)
model.config.forced_decoder_ids = None
model.config.suppress_tokens = []
print(f'Parameters: {model.num_parameters():,}')


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Parameters: 241,734,912


## 3. Preprocess (iterate with gc to save RAM)

In [ ]:
def prepare_example(ex):
    audio = ex['audio']
    arr = audio['array']
    sr = audio['sampling_rate']
    if sr != 16000:
        arr = librosa.resample(arr, orig_sr=sr, target_sr=16000)
    feat = feature_extractor(arr, sampling_rate=16000).input_features[0]
    labels = tokenizer(ex['transcription']).input_ids
    return {'input_features': feat, 'labels': labels}

train_list = []
for ex in train_iter:
    train_list.append(prepare_example(ex))
    del ex
gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

val_list = []
for ex in val_iter:
    val_list.append(prepare_example(ex))
    del ex
gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

train_dataset = Dataset.from_list(train_list)
val_dataset = Dataset.from_list(val_list)
del train_list, val_list
gc.collect()
print(f'Train: {len(train_dataset)}, Val: {len(val_dataset)}')

RuntimeError: Could not load libtorchcodec. Likely causes:
          1. FFmpeg is not properly installed in your environment. We support
             versions 4, 5, 6, 7, and 8, and we attempt to load libtorchcodec
             for each of those versions. Errors for versions not installed on
             your system are expected; only the error for your installed FFmpeg
             version is relevant. On Windows, ensure you've installed the
             "full-shared" version which ships DLLs.
          2. The PyTorch version (2.12.1+cpu) is not compatible with
             this version of TorchCodec. Refer to the version compatibility
             table:
             https://github.com/pytorch/torchcodec?tab=readme-ov-file#installing-torchcodec.
          3. Another runtime dependency; see exceptions below.

        The following exceptions were raised as we tried to load libtorchcodec:
        
[start of libtorchcodec loading traceback]
FFmpeg version 8:
Traceback (most recent call 

'The read operation timed out' thrown while requesting GET https://huggingface.co/datasets/google/WaxalNLP/resolve/e0a62aaebc61bd5bb8cac17a08d1b42c65551dd2/data/ASR/lin/lin-train-00000.parquet
Retrying in 1s [Retry 1/5].
'The read operation timed out' thrown while requesting GET https://huggingface.co/datasets/google/WaxalNLP/resolve/e0a62aaebc61bd5bb8cac17a08d1b42c65551dd2/data/ASR/sna/sna-train-00004.parquet
'The read operation timed out' thrown while requesting GET https://huggingface.co/datasets/google/WaxalNLP/resolve/e0a62aaebc61bd5bb8cac17a08d1b42c65551dd2/data/ASR/sna/sna-train-00003.parquet
'The read operation timed out' thrown while requesting GET https://huggingface.co/datasets/google/WaxalNLP/resolve/e0a62aaebc61bd5bb8cac17a08d1b42c65551dd2/data/ASR/lin/lin-train-00005.parquet
'The read operation timed out' thrown while requesting GET https://huggingface.co/datasets/google/WaxalNLP/resolve/e0a62aaebc61bd5bb8cac17a08d1b42c65551dd2/data/ASR/lin/lin-train-00002.parquet
'The re

In [ ]:
def data_collator(features):
    feat = torch.tensor(np.array([f['input_features'] for f in features]), dtype=torch.float)
    labels = [torch.tensor(f['labels']) for f in features]
    labels = torch.nn.utils.rnn.pad_sequence(labels, batch_first=True, padding_value=-100)
    return {'input_features': feat, 'labels': labels}

wer_metric = load_metric('wer')
cer_metric = load_metric('cer')

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = tokenizer.pad_token_id
    pred_str = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True)
    w = wer_metric.compute(predictions=pred_str, references=label_str)
    c = cer_metric.compute(predictions=pred_str, references=label_str)
    return {'wer': w, 'cer': c, 'combined': 0.5 * w + 0.5 * c}

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir='./whisper-waxal',
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=1e-5,
    warmup_steps=20,
    num_train_epochs=3,
    eval_strategy='steps',
    eval_steps=25,
    save_steps=50,
    logging_steps=5,
    report_to=['none'],
    dataloader_num_workers=0,
    dataloader_pin_memory=False,
    predict_with_generate=True,
    generation_max_length=225,
    fp16=torch.cuda.is_available(),
    save_total_limit=2,
    seed=42,
    data_seed=42,
    load_best_model_at_end=True,
    metric_for_best_model='combined',
    greater_is_better=False,
)

In [ ]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)
trainer.train()
trainer.save_model('./whisper-waxal-best')

## 4. Generate Submission

In [ ]:
best_model = WhisperForConditionalGeneration.from_pretrained('./whisper-waxal-best').to(device)
best_model.eval()

predictions = []
for i, example in enumerate(test_ds):
    audio = example['audio']
    arr = audio['array']
    sr = audio['sampling_rate']
    if sr != 16000:
        arr = librosa.resample(arr, orig_sr=sr, target_sr=16000)
    inputs = feature_extractor(arr, sampling_rate=16000, return_tensors='pt').input_features.to(device)
    with torch.no_grad():
        ids = best_model.generate(inputs)
    transcription = processor.batch_decode(ids, skip_special_tokens=True)[0]
    predictions.append({'id': example['id'], 'transcription': transcription})
    if (i + 1) % 100 == 0:
        print(f'  {i+1} done')

submission_df = pd.DataFrame(predictions)
submission_df.to_csv('../submissions/baseline_whisper_submission.csv', index=False)
print(f'Saved {len(submission_df)} predictions')